# Deepfake Detection — DFNet training on Kaggle GPU

**Before you run anything**, open the panel on the right and set:

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (or P100) |
| Internet | **On** (needed for `git clone`) |
| Input | Add dataset **`xhlulu/140k-real-and-fake-faces`** |

Then `Run All`. A full 25-epoch run takes roughly 2–3 hours, well inside
Kaggle's 9-hour GPU session limit and its 30 GPU-hours/week quota.

## 1. Confirm the GPU is actually attached

In [ ]:
import shutil, subprocess, torch

# shutil.which first: subprocess.run raises FileNotFoundError if the binary is
# absent, which is exactly the no-GPU case we are trying to report cleanly.
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("nvidia-smi not found - this session has no GPU attached")

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "No GPU attached.\n"
    "Fix: click 'Edit' to open the notebook editor, then in the right-hand\n"
    "sidebar set Session options -> Accelerator -> 'GPU T4 x2'.\n"
    "The session restarts; then Run All again.")

## 2. Get the project code

Cloned fresh from GitHub every run, so **push your local changes first** or
Kaggle will train an older version:

```bash
# on your Mac, before re-running this notebook
git add -A && git commit -m "your message" && git push
```

The repo must stay **public** for this clone to work without credentials.

In [ ]:
REPO_URL = "https://github.com/Preet1002/Deepfake_Detection.git"

import os, shutil, subprocess, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
PROJECT = WORKING / "Deepfake_Detection"

# Step out of PROJECT before deleting it. Re-running this cell would otherwise
# remove the process's own working directory (we chdir into it below), and git
# then fails with "Unable to read current working directory".
os.chdir(WORKING)
if PROJECT.exists():
    shutil.rmtree(PROJECT)          # always start from a clean clone

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)                   # `!python` cells below inherit this cwd
sys.path.insert(0, str(PROJECT))
print("working dir:", os.getcwd())
print(sorted(p.name for p in PROJECT.iterdir() if not p.name.startswith(".")))

## 3. Locate the dataset mount

Kaggle mounts inputs read-only, so we point the loader straight at it. The
dataset ships `train/valid/test`; the loader accepts `valid` as the val split,
so **no copying or preparation is needed**.

This searches `/kaggle/input` for *any* folder shaped like split/real+fake, so
it works with re-uploaded mirrors of the dataset, not just one exact slug.

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
SPLIT_NAMES = {"train", "training", "valid", "validation", "val", "test", "testing"}


def subdirs(path):
    try:
        return [c for c in path.iterdir() if c.is_dir()]
    except (PermissionError, OSError):
        return []


def usable_splits(path):
    """Split folders under `path` that actually contain both real/ and fake/."""
    return [c for c in subdirs(path)
            if c.name.lower() in SPLIT_NAMES
            and {d.name.lower() for d in subdirs(c)} >= {"real", "fake"}]


def is_dataset_root(path):
    return len(usable_splits(path)) >= 2


def find_roots(start, max_depth=8):
    """Breadth-first search, never descending into the image folders.

    Depth has to be generous: Kaggle nests some mounts as
    /kaggle/input/datasets/<owner>/<slug>/... which puts the real root six
    levels down. This stays cheap because we stop as soon as a directory looks
    like a dataset root and never call iterdir() inside real/ or fake/.
    """
    found, frontier = [], [(start, 0)]
    while frontier:
        path, depth = frontier.pop(0)
        if is_dataset_root(path):
            found.append(path)
            continue                       # no need to look deeper here
        if depth < max_depth:
            frontier += [(c, depth + 1) for c in subdirs(path)
                         if c.name.lower() not in {"real", "fake"}]
    return found


def print_tree(path, prefix="", depth=0, max_depth=5):
    for child in sorted(subdirs(path))[:12]:
        print(f"{prefix}  {child.name}/")
        if depth < max_depth:
            print_tree(child, prefix + "  ", depth + 1, max_depth)


roots = find_roots(INPUT)
if not roots:
    print("No dataset found. Directory tree under /kaggle/input:")
    print_tree(INPUT)
    raise SystemExit(
        "Add a real/fake face dataset via '+ Add Input'. Paste this URL into "
        "the search box: "
        "https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces")

# ---------------------------------------------------------------------------
# Which dataset to TRAIN on, matched as a substring of its path. This must be
# explicit: once a second dataset is mounted for cross-dataset evaluation,
# picking "the first root found" silently depends on filesystem order, and a
# run can end up training on the wrong dataset - which invalidates any
# comparison against earlier runs.
TRAIN_ON = "real-vs-fake"          # the 140k StyleGAN dataset
# ---------------------------------------------------------------------------

# Prefer the copy with the most usable splits: a mirror with only train/test
# would leave training with no validation set to select checkpoints on.
roots.sort(key=lambda r: len(usable_splits(r)), reverse=True)
if len(roots) > 1:
    print("Datasets mounted:")
    for r in roots:
        print(f"   {len(usable_splits(r))} splits  {r}")

chosen = [r for r in roots if TRAIN_ON.lower() in str(r).lower()]
if not chosen:
    raise SystemExit(
        f"No mounted dataset matches TRAIN_ON={TRAIN_ON!r}.\n"
        f"Roots found: {[str(r) for r in roots]}\n"
        f"Edit TRAIN_ON above to match the one you mean to train on.")

DATA_ROOT = str(chosen[0])
names = {c.name.lower() for c in usable_splits(Path(DATA_ROOT))}
print("dataset root:", DATA_ROOT)
for split in sorted(usable_splits(Path(DATA_ROOT))):
    counts = {c.name: sum(1 for _ in c.iterdir()) for c in sorted(subdirs(split))}
    print(f"  {split.name:11s} {counts}")

if not names & {"val", "valid", "validation"}:
    raise SystemExit(
        "\nThis copy has no validation split, which training needs to select "
        "the best checkpoint.\nUse the original dataset instead - paste this "
        "into '+ Add Input':\n"
        "  https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces")

## 4. Write the run config

Built from `configs/kaggle.yaml` with the detected dataset path patched in.

Two switches live at the top of the cell:

| Switch | Effect |
|---|---|
| `ABLATION = "no_srm"` | trains the RGB-only baseline instead of the two-stream model |
| `MULTI_SOURCE = True` | trains on **every** mounted dataset at once, capped per source |

`MULTI_SOURCE` is the fix for the cross-dataset collapse: a model trained on a
single dataset learns that dataset's build pipeline, and scores at or below
chance on any other one. Mount both datasets before turning it on.


In [ ]:
import yaml
from pathlib import Path

# ---------------------------------------------------------------------------
# Set ABLATION = "no_srm" to train the RGB-only baseline instead of the full
# two-stream model. Run it once each way and compare - that contrast is the
# headline experiment for the report.
ABLATION = None                    # None | "no_srm" | "plain" | "patch"

# Train on EVERY mounted dataset at once instead of just TRAIN_ON. A model that
# only ever sees one dataset's build pipeline learns that pipeline, which is
# exactly the cross-dataset collapse the single-source runs showed. Capped per
# source so the larger dataset cannot drown out the smaller one.
MULTI_SOURCE = False               # True | False
LIMIT_PER_CLASS = 25000            # per source, per class, when MULTI_SOURCE

# Heavier degradation augmentation. A detector trained on pristine generator
# output locks onto that generator's high-frequency fingerprint, which is
# exactly what fails on an unseen generator. Blurring and re-compressing hard
# during training destroys that fingerprint, forcing the model onto cues that
# survive - and those transfer better. (Wang et al., CVPR 2020, found this the
# single most effective intervention for cross-generator transfer.)
HEAVY_AUG = False                  # True | False
# ---------------------------------------------------------------------------

config = yaml.safe_load(Path("configs/kaggle.yaml").read_text())

if MULTI_SOURCE:
    # Every source needs its own train AND val split: build_dataloaders
    # resolves each split under each root and raises if one is missing, which
    # would fail minutes into the run rather than here.
    def split_names(root):
        return {c.name.lower() for c in usable_splits(root)}

    usable = [r for r in roots
              if split_names(r) & {"train", "training"}
              and split_names(r) & {"val", "valid", "validation"}]
    if len(usable) < 2:
        raise SystemExit(
            f"MULTI_SOURCE needs two or more datasets that each have train and "
            f"val splits; usable: {[str(r) for r in usable]}.\n"
            f"Add another via '+ Add Input', e.g. "
            f"manjilkarki/deepfake-and-real-images.")
    TRAIN_ROOTS = [str(r) for r in usable]
    config["data"]["root"] = TRAIN_ROOTS
    config["data"]["limit_per_class"] = LIMIT_PER_CLASS
else:
    TRAIN_ROOTS = [DATA_ROOT]
    config["data"]["root"] = DATA_ROOT

if HEAVY_AUG:
    config["aug"].update(
        jpeg=0.5, jpeg_quality=[30, 100],       # was [40, 95]
        blur=0.5, blur_sigma=[0.0, 3.0],        # was p=0.2, sigma up to 1.2
        downscale=0.5, downscale_range=[0.25, 0.9],
    )

RUN_NAME = "dfnet" if ABLATION is None else f"dfnet_{ABLATION}"
if MULTI_SOURCE:
    RUN_NAME += "_multi"
if HEAVY_AUG:
    RUN_NAME += "_heavyaug"
RUN_DIR = f"/kaggle/working/runs/{RUN_NAME}"
config["train"]["out_dir"] = RUN_DIR
if ABLATION == "no_srm":
    config["model"]["use_srm"] = False
elif ABLATION == "plain":
    # Comparison baseline: a conventional single-stream CNN of ResNet-18 shape,
    # trained from scratch under an identical protocol. Deliberately LARGER than
    # DFNet (11.2M parameters against 8.0M), so a DFNet win cannot be waved away
    # as extra capacity.
    config["model"].update(use_srm=False, se_ratio=0.0,
                           stage_channels=[64, 128, 256, 512])
elif ABLATION == "patch":
    # One stage fewer, so the final receptive field covers a patch rather than
    # the whole face. Local texture is reported to transfer across generators
    # better than global composition (Chai et al., ECCV 2020).
    config["model"].update(stage_channels=[64, 128, 256],
                           blocks_per_stage=[2, 2, 2])

Path("/kaggle/working/configs").mkdir(parents=True, exist_ok=True)

# A capped copy for the shakedown run, and the real one for the full run.
smoke = yaml.safe_load(yaml.safe_dump(config))
smoke["data"]["limit_per_class"] = 2000
smoke["train"].update(epochs=2, warmup_epochs=0,
                      out_dir="/kaggle/working/runs/smoke")
Path("/kaggle/working/configs/smoke.yaml").write_text(yaml.safe_dump(smoke))
Path("/kaggle/working/configs/full.yaml").write_text(yaml.safe_dump(config))

print("run dir:", RUN_DIR)
print("training on:")
for r in TRAIN_ROOTS:
    print("  ", r)
print()
print(yaml.safe_dump(config, sort_keys=False))


## 5. Shakedown run (~4 minutes)

2 epochs on 2k images per class. The numbers are meaningless — this only proves
the CUDA path, the data mount and the checkpointing all work before you commit
to a multi-hour run. **The local development was verified on Apple MPS, so this
is the first real exercise of the CUDA + AMP path.**

In [ ]:
import gc, importlib, os, sys, torch

# Run training IN-PROCESS rather than via `!python`. A subprocess writes to a
# pipe, so Python block-buffers its stdout and the cell looks frozen for
# minutes; worse, if it dies early the error can vanish instead of surfacing.
# In-process, prints stream immediately and any failure is a real traceback.
print("cwd:", os.getcwd())
print("src/ present:", os.path.isdir("src"))

import src.train
importlib.reload(src.train)                 # pick up a fresh clone on re-runs

src.train.main(["--config", "/kaggle/working/configs/smoke.yaml"])

gc.collect()
torch.cuda.empty_cache()                    # free VRAM before the full run
print("\nGPU memory now:",
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## 6. Full training run (~2–3 hours)

Watch the first epoch's timing. If it is much slower than ~6 min, the 4 vCPUs
are bottlenecking on JPEG augmentation rather than the GPU — lower `aug.jpeg`
to `0.3` in the config and re-run.

In [ ]:
import gc, torch

src.train.main(["--config", "/kaggle/working/configs/full.yaml"])

gc.collect()
torch.cuda.empty_cache()

## 7. Evaluate on the held-out test split

Scored in full, even though training used the capped subset in the smoke run.

In [ ]:
import src.evaluate
from pathlib import Path

# With MULTI_SOURCE there is one in-domain test set per training source. The
# first keeps the plain eval_test/ path so the figure cells below still find
# their images.
for i, root in enumerate(TRAIN_ROOTS):
    out_dir = f"{RUN_DIR}/eval_test" if i == 0 \
        else f"{RUN_DIR}/eval_test_{Path(root).name}"
    if len(TRAIN_ROOTS) > 1:
        print(f"\n===== IN-DOMAIN: {root} =====")
    src.evaluate.main([
        "--checkpoint", f"{RUN_DIR}/best.pt",
        "--data-root", root,
        "--split", "test",
        "--batch-size", "256",
        "--out-dir", out_dir,
    ])


## 7b. Cross-dataset evaluation — the number that actually matters

Scores the same checkpoint on every *other* real/fake dataset mounted under
`/kaggle/input`. Same-dataset accuracy mostly measures whether the model
memorised one dataset's build pipeline; this measures whether it learned
anything about synthesis.

Add a second dataset via **+ Add Input** to populate this, e.g.
`manjilkarki/deepfake-and-real-images`. Skipped silently if there isn't one.

In [ ]:
# Only datasets the model never trained on count as cross-dataset. Under
# MULTI_SOURCE every mounted set is in TRAIN_ROOTS, so this correctly reports
# that there is nothing held out rather than quietly re-scoring training data.
others = [r for r in find_roots(INPUT) if str(r) not in TRAIN_ROOTS]

if not others:
    if len(TRAIN_ROOTS) > 1:
        print("Every mounted dataset was trained on, so there is no held-out "
              "set left to measure generalisation with.\nMount a third dataset "
              "and leave it out of training for an honest number.")
    else:
        print("No second dataset mounted - skipping. Add one via '+ Add Input' "
              "to measure generalisation.")
else:
    for other in others:
        print(f"\n===== CROSS-DATASET: {other} =====")
        try:
            src.evaluate.main([
                "--checkpoint", f"{RUN_DIR}/best.pt",
                "--data-root", str(other),
                "--split", "test",
                "--batch-size", "256",
                "--out-dir", f"{RUN_DIR}/eval_cross_{other.name}",
            ])
        except FileNotFoundError as exc:
            # Some mirrors have no test split; fall back to validation.
            print(f"  no test split ({exc}); trying valid")
            src.evaluate.main([
                "--checkpoint", f"{RUN_DIR}/best.pt",
                "--data-root", str(other),
                "--split", "val",
                "--batch-size", "256",
                "--out-dir", f"{RUN_DIR}/eval_cross_{other.name}",
            ])


## 8. Training curves and test figures

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

history = json.loads(Path(f"{RUN_DIR}/history.json").read_text())
epochs = [h["epoch"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss"); axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(epochs, [h["auc"] for h in history], label="val AUC")
axes[1].plot(epochs, [h["accuracy"] for h in history], label="val accuracy")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("Validation")
plt.tight_layout(); plt.show()

best = max(history, key=lambda h: h["auc"])
print(f"best epoch {best['epoch']}: AUC={best['auc']:.4f} acc={best['accuracy']:.4f}")

In [ ]:
from IPython.display import Image as IPyImage, display

for name in ("roc.png", "confusion_matrix.png"):
    path = f"{RUN_DIR}/eval_test/{name}"
    if Path(path).exists():
        display(IPyImage(filename=path))

## 9. Grad-CAM on a few test images

Warm regions are what pushed the decision towards FAKE.

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from src.predict import Detector

detector = Detector(f"{RUN_DIR}/best.pt")
random.seed(0)

# Resolve split/class folders case-insensitively: datasets ship either
# test/real or Test/Real, and a hardcoded lowercase path finds nothing on
# Linux, leaving random.sample() to fail on an empty list.
from src.data.dataset import IMAGE_EXTENSIONS, resolve_class_dir, resolve_split_dir

test_dir = resolve_split_dir(DATA_ROOT, "test")
samples = []
for class_name in ("real", "fake"):
    folder = resolve_class_dir(test_dir, class_name)
    if folder is None:
        print(f"no {class_name}/ under {test_dir}, skipping")
        continue
    paths = sorted(p for p in folder.iterdir()
                   if p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        print(f"no images in {folder}, skipping")
        continue
    samples += [(p, class_name) for p in random.sample(paths, min(3, len(paths)))]

if not samples:
    raise SystemExit(f"No test images found under {test_dir}")

fig, axes = plt.subplots(2, len(samples), figsize=(3 * len(samples), 6.5))
for col, (path, truth) in enumerate(samples):
    image = Image.open(path).convert("RGB")
    # These are already tight face crops, so skip detection.
    result = detector.predict(image, detect_faces=False, explain=True)
    axes[0][col].imshow(image); axes[0][col].axis("off")
    axes[0][col].set_title(f"truth: {truth}", fontsize=10)
    axes[1][col].imshow(result.faces[0].heatmap_image); axes[1][col].axis("off")
    correct = result.label.lower() == truth
    axes[1][col].set_title(f"{result.label} p={result.fake_probability:.2f} "
                           f"{'OK' if correct else 'WRONG'}",
                           fontsize=10, color="green" if correct else "red")
plt.tight_layout(); plt.show()

## 10. Package the results for download

Everything under `/kaggle/working/` is kept as notebook output (20 GB limit).

This also bundles everything into a single **`dfnet_results.zip`**. Download that
rather than the individual files: browsers preview `.json` and `.png` in a new
tab instead of saving them, but always download a `.zip`.

Drop `best.pt` into `runs/dfnet/` on your Mac — the checkpoint carries its own
config, so `evaluate.py` and the web app load it unchanged.

In [ ]:
import os, shutil
from pathlib import Path

WORKING = Path("/kaggle/working")
os.chdir(WORKING)          # never delete the directory we are standing in

shutil.rmtree(WORKING / "runs/smoke", ignore_errors=True)      # useless 300 KB model
shutil.rmtree(WORKING / "Deepfake_Detection", ignore_errors=True)   # re-cloneable

# One archive downloads cleanly; individual .json/.png open as previews instead.
archive = shutil.make_archive(str(WORKING / "dfnet_results"), "zip",
                              root_dir=str(WORKING), base_dir="runs")
print("bundle:", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)\n")

for p in sorted(WORKING.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(WORKING)}")